# Ekstraksi Fitur Klasik Ringkas & Terbaik: Klasifikasi Kerapian Sandal (Rapi vs Berantakan)

Notebook ini mengekstrak **34 fitur klasik terbaik dan paling representatif** dari 6 kategori utama pengolahan citra (dipangkas dari ~200+ fitur agar bebas redundansi, cepat diproses, dan ideal untuk dataset 165 citra):

1. **Warna (9 fitur)**: `mean`/`std` RGB (`Red, Green, Blue`), `mean_gray`, `std_gray`, dan `color_ccv_coherent_ratio` (keutuhan blok warna).
2. **Tekstur (7 fitur)**: `glcm_contrast`, `glcm_homogeneity`, `glcm_energy`, `glcm_correlation`, `glcm_entropy`, `lbp_uniform_ratio`, dan `gabor_orientation_dominance`.
3. **Tepi & Gradien (6 fitur)**: `hog_entropy`, `hog_dominant_ratio`, `edge_laplacian_var`, `edge_canny_density`, `edge_canny_spatial_spread`, dan `edge_canny_num_contours`.
4. **Bentuk & Geometri (6 fitur)**: `shape_area_ratio`, `shape_solidity` (*Convex Hull*), `shape_circularity`, `shape_eccentricity`, `shape_num_objects`, dan `hu_moment_1`.
5. **Titik Kunci (3 fitur)**: `kp_density` (*Harris Corners*), `kp_spatial_spread`, dan `kp_desc_diversity`.
6. **Domain Frekuensi (3 fitur)**: `fft_high_energy_ratio`, `fft_spectral_entropy`, dan `dwt_detail_energy_ratio` (*Haar Wavelet Level-1*).

---

In [ ]:
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11


---
## 1. Pemeriksaan Direktori Dataset & Verifikasi 2 Kelas

Mendeteksi lokasi project secara otomatis (baik dijalankan dari folder `code/` maupun dari root project) serta memverifikasi kelas `Rapi` dan `Berantakan`.

In [ ]:
cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd / "uas" / "ProjectImageClasification",
    cwd / "uas" / "SandalClasification",
]
PROJECT_DIR = next((p for p in candidates if (p / "Data").exists()), cwd.parent)
DATASET_DIR = PROJECT_DIR / "Data"
MODULE_PATH = PROJECT_DIR / "code" / "ekstraksi_fitur.py"

# Muat modul secara eksplisit agar tidak bentrok dengan stdlib 'code'
spec = importlib.util.spec_from_file_location("sandal_ekstraksi_fitur", MODULE_PATH)
ekstraksi_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ekstraksi_mod)

print("Project Dir :", PROJECT_DIR.resolve())
print("Dataset Dir :", DATASET_DIR.resolve())

expected_classes = ["Rapi", "Berantakan"]
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_counts = {}

for cls in expected_classes:
    cls_path = DATASET_DIR / cls
    assert cls_path.exists(), f"Direktori kelas '{cls}' TIDAK ditemukan di {DATASET_DIR}"
    files = [f for f in cls_path.iterdir() if f.is_file() and f.suffix.lower() in valid_exts]
    class_counts[cls] = len(files)
    print(f"[V] Kelas '{cls}': Ditemukan {len(files)} file gambar.")

print(f"Total seluruh gambar dataset: {sum(class_counts.values())} gambar.")


---
## 2. Visualisasi Representasi Fitur pada Sampel Citra

Perbandingan visual antara sampel **Rapi** dan **Berantakan** pada tahap RGB, Segmentasi Otsu (Bentuk), Tepi Canny, Gradien Sobel (HOG), Titik Kunci Harris, dan Spektrum 2D FFT.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for row_idx, cls_name in enumerate(expected_classes):
    cls_dir = DATASET_DIR / cls_name
    sample_path = sorted(
        [f for f in cls_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_exts],
        key=lambda p: p.name,
    )[0]

    with Image.open(sample_path) as img:
        img_std = img.convert("RGB").resize(ekstraksi_mod.STANDARD_SIZE, resample=ekstraksi_mod.RESAMPLE_BILINEAR)
        arr_std = np.array(img_std, dtype=np.float64)

    gray = 0.299 * arr_std[:, :, 0] + 0.587 * arr_std[:, :, 1] + 0.114 * arr_std[:, :, 2]
    g_norm = gray / 255.0

    mask_otsu = ekstraksi_mod._otsu_foreground_mask(gray)
    edges_canny = ekstraksi_mod._canny_like_edges(g_norm)
    gx = ndimage.sobel(g_norm, axis=1)
    gy = ndimage.sobel(g_norm, axis=0)
    mag_sobel = np.hypot(gx, gy)

    sxx = ndimage.gaussian_filter(gx * gx, sigma=1.5)
    syy = ndimage.gaussian_filter(gy * gy, sigma=1.5)
    sxy = ndimage.gaussian_filter(gx * gy, sigma=1.5)
    R = (sxx * syy - sxy * sxy) - 0.04 * ((sxx + syy) ** 2)
    kp_mask = (R == ndimage.maximum_filter(R, size=5)) & (R > 0.01 * np.max(R))
    ky, kx = np.where(kp_mask)

    fft_mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g_norm - g_norm.mean()))))

    axes[row_idx, 0].imshow(arr_std.astype(np.uint8))
    axes[row_idx, 0].set_title(f"{cls_name}: RGB\n({sample_path.name})")
    axes[row_idx, 1].imshow(mask_otsu, cmap="gray")
    axes[row_idx, 1].set_title("Mask Bentuk (Otsu)")
    axes[row_idx, 2].imshow(edges_canny, cmap="gray")
    axes[row_idx, 2].set_title("Tepi Canny")
    axes[row_idx, 3].imshow(mag_sobel, cmap="inferno")
    axes[row_idx, 3].set_title("Gradien Sobel (HOG)")
    axes[row_idx, 4].imshow(gray, cmap="gray")
    axes[row_idx, 4].scatter(kx, ky, c="red", s=10, alpha=0.8)
    axes[row_idx, 4].set_title(f"Harris Keypoints ({len(kx)})")
    axes[row_idx, 5].imshow(fft_mag, cmap="viridis")
    axes[row_idx, 5].set_title("Spektrum 2D FFT")

    for col in range(6):
        axes[row_idx, col].axis("off")

plt.tight_layout()
plt.show()


---
## 3. Ekstraksi 34 Fitur Klasik Terbaik & Simpan ke CSV

Menjalankan `proses_dataset()` untuk mengekstrak 34 fitur terbaik pada seluruh gambar dan menyimpannya ke `hasil_ekstraksi/fitur_ekstraksi_sandal.csv`.

In [ ]:
OUTPUT_DIR = PROJECT_DIR / "hasil_ekstraksi"
df_features, csv_output_path = ekstraksi_mod.proses_dataset(
    dataset_dir=DATASET_DIR,
    output_dir=OUTPUT_DIR,
)

meta_cols = ["filename", "label", "width", "height"]
feature_cols = [c for c in df_features.columns if c not in meta_cols]

def get_category(col_name: str) -> str:
    if col_name.startswith(("mean_", "std_", "color_")):
        return "1. Warna"
    if col_name.startswith(("glcm_", "lbp_", "gabor_")):
        return "2. Tekstur"
    if col_name.startswith(("hog_", "edge_")):
        return "3. Tepi & Gradien"
    if col_name.startswith(("shape_", "hu_")):
        return "4. Bentuk & Geometri"
    if col_name.startswith("kp_"):
        return "5. Titik Kunci"
    if col_name.startswith(("fft_", "dwt_")):
        return "6. Frekuensi"
    return "Lainnya"

cat_series = pd.Series({col: get_category(col) for col in feature_cols})
print("\n=== RINGKASAN JUMLAH FITUR PER KATEGORI ===")
print(cat_series.value_counts().sort_index())
print(f"Total Fitur Ringkas: {len(feature_cols)} fitur dari {len(df_features)} citra.")
df_features.head()


---
## 4. Evaluasi Klasifikasi & Seleksi Fitur Terbaik (Top-15)

Membandingkan akurasi *5-Fold Cross-Validation* antara seluruh **34 fitur ringkas** vs subset **Top-15 fitur terpilih** (*ANOVA F-Score* + *Random Forest Importance*), serta menyimpan subset terpilih ke `hasil_ekstraksi/fitur_terpilih_sandal.csv`.

In [ ]:
X = df_features[feature_cols].fillna(0.0)
y = df_features["label"]

n_splits = min(5, int(y.value_counts().min()))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Peringkat ANOVA F-Score & Random Forest Importance
f_vals, p_vals = f_classif(X, y)
f_vals = np.nan_to_num(f_vals, nan=0.0)
p_vals = np.nan_to_num(p_vals, nan=1.0)

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X, y)

df_importance = pd.DataFrame({
    "feature": feature_cols,
    "category": [cat_series[c] for c in feature_cols],
    "rf_importance": rf.feature_importances_,
    "anova_f_score": f_vals,
    "p_value": p_vals,
}).sort_values("rf_importance", ascending=False).reset_index(drop=True)

top15_features = df_importance.head(15)["feature"].tolist()
df_selected = df_features[["filename", "label"] + top15_features]
selected_csv_path = OUTPUT_DIR / "fitur_terpilih_sandal.csv"
df_selected.to_csv(selected_csv_path, index=False)
print(f"Subset Top-15 fitur terbaik disimpan ke: {selected_csv_path}")

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42)),
}

print(f"\n=== PERBANDINGAN {n_splits}-FOLD CROSS-VALIDATION ===")
for label_set, X_sub in [(f"34 Fitur Ringkas", X), ("Top-15 Fitur Terpilih", X[top15_features])]:
    print(f"\n[{label_set}]")
    for model_name, clf in models.items():
        scores = cross_val_score(clf, X_sub, y, cv=cv, scoring="accuracy")
        print(f"  {model_name:22s}: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)")

df_importance.head(15)


---
## 5. Visualisasi Top-15 Fitur Terbaik & Perbandingan 6 Kategori

In [ ]:
top15 = df_importance.head(15)
plt.figure(figsize=(12, 6.5))
sns.barplot(data=top15, y="feature", x="rf_importance", hue="category", dodge=False)
plt.title("Top 15 Fitur Klasik Terbaik (Rapi vs Berantakan)")
plt.xlabel("Random Forest Feature Importance")
plt.ylabel("Nama Fitur")
plt.legend(title="Kategori", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

categories_sorted = sorted(cat_series.unique())[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, cat_name in zip(axes.ravel(), categories_sorted):
    best_feat = (
        df_importance[df_importance["category"] == cat_name]
        .sort_values("anova_f_score", ascending=False)
        .iloc[0]["feature"]
    )
    sns.boxplot(
        data=df_features,
        x="label",
        y=best_feat,
        hue="label",
        legend=False,
        palette={"Rapi": "#4CAF50", "Berantakan": "#F44336"},
        ax=ax,
    )
    ax.set_title(f"{cat_name}\n({best_feat})", fontsize=11)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()


---
## 6. Kesimpulan

1. **Fitur Ringkas & Bebas Redundansi**: Jumlah fitur diringkas dari ~206 fitur menjadi **34 fitur terbaik** lintas 6 kategori (Warna, Tekstur, Tepi/Gradien, Bentuk/Geometri, Keypoint, dan Frekuensi) yang seimbang dengan ukuran dataset.
2. **Output Tersimpan**:
   - `hasil_ekstraksi/fitur_ekstraksi_sandal.csv` (34 fitur klasik terbaik)
   - `hasil_ekstraksi/fitur_terpilih_sandal.csv` (subset Top-15 fitur paling diskriminatif)